# ESM3/C

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import os
# only load this one time per session
if 'NOTEBOOK_INITIALIZED' not in globals():
    os.chdir(os.path.dirname(os.path.abspath('.')))
    NOTEBOOK_INITIALIZED = True

import src.utils as utils
import src.config as config
import src.haplosaurus as hs

# import src.vep_pipeline as vp
import src.vep_metrics as vm
import src.ESM3 as esm3

import torch
from tqdm import tqdm


/home/schilder/.conda/envs/esm3/lib/python3.12/site-packages/Bio/Application/__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


## ESM3

In [3]:
devices = vm.get_available_gpus()
devices

[(0, 84974239744), (1, 84974239744), (2, 84974239744), (3, 84974239744)]

In [4]:
import torch
devices = []
for i in range(torch.cuda.device_count()):
   devices.append(torch.cuda.get_device_properties(i))
devices

[_CudaDeviceProperties(name='NVIDIA A100 80GB PCIe', major=8, minor=0, total_memory=81037MB, multi_processor_count=108, uuid=57a5dbce-e7ee-766b-1803-e35688317a0c, L2_cache_size=40MB),
 _CudaDeviceProperties(name='NVIDIA A100 80GB PCIe', major=8, minor=0, total_memory=81037MB, multi_processor_count=108, uuid=ee601dc3-6f2c-3e1b-8b01-5c4012c77075, L2_cache_size=40MB),
 _CudaDeviceProperties(name='NVIDIA A100 80GB PCIe', major=8, minor=0, total_memory=81037MB, multi_processor_count=108, uuid=d1d17271-1e44-abb0-9cf6-de92acf10943, L2_cache_size=40MB),
 _CudaDeviceProperties(name='NVIDIA A100 80GB PCIe', major=8, minor=0, total_memory=81037MB, multi_processor_count=108, uuid=ca14fd86-8f12-0e67-15c5-eefe075aa2dd, L2_cache_size=40MB)]

In [2]:
import torch
torch.cuda.set_device(1)

In [ ]:
from huggingface_hub import login   
from esm.models.esm3 import ESM3
from esm.sdk.api import ESM3InferenceClient, ESMProtein, GenerationConfig


# Will instruct you how to get an API key from huggingface hub, make one with "Read" permission.
# login( token="{HUGGINGFACE_TOKEN}")

# This will download the model weights and instantiate the model on your machine.
model: ESM3InferenceClient = ESM3.from_pretrained("esm3-open").to("cuda")  # Use GPU 2 which has the most available memory

# # Generate a completion for a partial Carbonic Anhydrase (2vvb)
prompt = "___________________________________________________DQATSLRILNNGHAFNVEFDDSQDKAVLKGGPLDGTYRLIQFHFHWGSLDGQGSEHTVDKKKYAAELHLVHWNTKYGDFGKAVQQPDGLAVLGIFLKVGSAKPGLQKVVDVLDSIKTKGKSADFTNFDPRGLLPESLDYWTYPGSLTTPP___________________________________________________________"
protein = ESMProtein(sequence=prompt)

# Generate the sequence, then the structure. This will iteratively unmask the sequence track.
protein = model.generate(protein, GenerationConfig(track="sequence", num_steps=8, temperature=0.7))

# We can show the predicted structure for the generated sequence.
protein = model.generate(protein, GenerationConfig(track="structure", num_steps=8))
protein.to_pdb("./tmp/generation.pdb")

# Then we can do a round trip design by inverse folding the sequence and recomputing the structure
protein.sequence = None
protein = model.generate(protein, GenerationConfig(track="sequence", num_steps=8))
protein.coordinates = None
protein = model.generate(protein, GenerationConfig(track="structure", num_steps=8))
protein.to_pdb("./tmp/round_tripped.pdb")

100%|██████████| 8/8 [00:00<00:00, 20.77it/s]
/home/schilder/.conda/envs/esm3/lib/python3.12/site-packages/esm/utils/structure/protein_complex.py:223: UserWarning: Entity ID not found in metadata, using None as default
  warnings.warn("Entity ID not found in metadata, using None as default")
/home/schilder/.conda/envs/esm3/lib/python3.12/site-packages/esm/models/vqvae.py:286: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=False):  # type: ignore
100%|██████████| 8/8 [00:00<00:00, 20.37it/s]


In [22]:
import src.ESM3 as ESM3

model, alphabet = ESM3.load_model(model_loc="esm3_sm_open_v1")


In [6]:
import src.ESM_predict as ESMp

sequence = "MQATSLRILNNGHAFNVEFDDSQDKAVLKGGPLDGTYRLIQFHFHWGSLDGQGSEHTVDKKKYAAELHLVHWNTKYGDFGKAVQQPDGLAVLGIFLKVGSAKPGLQKVVDVLDSIKTKGKSADFTNFDPRGLLPESLDYWTYPGSLTTPP"
mutations = ["Q2A","S5T","L10A"] 
batch_tokens, protein_tensor = ESM3.tokenize_sequence(model, sequence) 


NameError: name 'ESM3' is not defined

In [11]:
import src.ESM_predict as ESMp

ESMp.seq_to_batch(sequence, alphabet)

(['protein1'],
 ['MQATSLRILNNGHAFNVEFDDSQDKAVLKGGPLDGTYRLIQFHFHWGSLDGQGSEHTVDKKKYAAELHLVHWNTKYGDFGKAVQQPDGLAVLGIFLKVGSAKPGLQKVVDVLDSIKTKGKSADFTNFDPRGLLPESLDYWTYPGSLTTPP'],
 tensor([[ 0, 20, 16,  5, 11,  8,  4, 10, 12,  4, 17, 17,  6, 21,  5, 18, 17,  7,
           9, 18, 13, 13,  8, 16, 13, 15,  5,  7,  4, 15,  6,  6, 14,  4, 13,  6,
          11, 19, 10,  4, 12, 16, 18, 21, 18, 21, 22,  6,  8,  4, 13,  6, 16,  6,
           8,  9, 21, 11,  7, 13, 15, 15, 15, 19,  5,  5,  9,  4, 21,  4,  7, 21,
          22, 17, 11, 15, 19,  6, 13, 18,  6, 15,  5,  7, 16, 16, 14, 13,  6,  4,
           5,  7,  4,  6, 12, 18,  4, 15,  7,  6,  8,  5, 15, 14,  6,  4, 16, 15,
           7,  7, 13,  7,  4, 13,  8, 12, 15, 11, 15,  6, 15,  8,  5, 13, 18, 11,
          17, 18, 13, 14, 10,  6,  4,  4, 14,  9,  8,  4, 13, 19, 22, 11, 19, 14,
           6,  8,  4, 11, 11, 14, 14,  2]]))

In [ ]:
import os
# only load this one time per session
if 'NOTEBOOK_INITIALIZED' not in globals():
    os.chdir(os.path.dirname(os.path.abspath('.')))
    NOTEBOOK_INITIALIZED = True

import src.vep_pipeline as vp 
import src.haplosaurus as hs
import torch
torch.cuda.set_device("cuda:2")

save_paths = vp.vep_pipeline(
    hap_dir = hs.DIR_DICT["HGDP_haplotypes"],
    # prot_df = prot_df,
    # haplotypes = haplotypes, 
    models = [ 
            "esm3_sm_open_v1"
             ],
    scoring_strategies = [
        # "wt-marginals", 
        "masked-marginals", 
        # "pseudo-ppl" # Takes much longer to run
        ],
    enable_data_parallel=False,
    source_types = ["clinical_ProteinGym_substitutions"],
    verbose = False
)

print(len(save_paths),"results files generated.")

Found haplotypes of 85520 transcripts in: '/home/schilder/projects/data/Human_Genome_Diversity_Project/haplosaurus/'


Adding reference haplotype:   0%|          | 0/2652 [00:00<?, ?it/s]

Getting haplotype sequences:   0%|          | 0/2652 [00:00<?, ?it/s]

Getting haplotype names:   0%|          | 0/2652 [00:00<?, ?it/s]

Processing models:   0%|          | 0/1 [00:00<?, ?it/s]

...

Computing token probabilities: 'masked-marginals':   0%|          | 0/600 [00:00<?, ?it/s]

Computing 'masked-marginals' for ESM3:   0%|          | 0/11 [00:00<?, ?it/s]

Processing scoring strategies:   0%|          | 0/1 [00:00<?, ?it/s]

Computing token probabilities: 'masked-marginals':   0%|          | 0/600 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [55]:
def _check_token_probs(token_probs,
                           batch_tokens,
                           alphabet):
        assert token_probs.shape[0] == 1, f"Token probabilities must have a batch dimension of 1. Got {token_probs.shape[0]}."
        assert token_probs.shape[1] == batch_tokens[0].shape[0], f"Token probabilities must have a sequence length of {batch_tokens[0].shape[0]}. Got {token_probs.shape[1]}."
        assert token_probs.shape[2] == len(alphabet), f"Token probabilities must have a vocabulary size of {len(alphabet)}. Got {token_probs.shape[2]}."

import attr
from esm.sdk.api import LogitsConfig, ESMProtein

if sequence is None:
    raise ValueError("Sequence must be provided for masked-marginals-esm3")

progress_bar = True
leave = False
token_indices = [0,10,100,200]
        
batch_tokens = batch_tokens.cuda()
# Get the device from batch_tokens
device = batch_tokens.device

# Construct the protein tensor
protein_tensor = model.encode( ESMProtein(sequence=sequence))

# Get mask token ID based on model type
# For ESM-C, mask token is typically 32
mask_token_id = getattr(model, "mask_token_id", 32)

# If using ESM3, get mask token ID from tokenizers
if  hasattr(model, "tokenizers"):
    mask_token_id = model.tokenizers.sequence.mask_token_id 

all_token_probs = []
for i in tqdm(range(batch_tokens.size(-1)),
                desc=f"Computing token probabilities: 'masked-marginals'",
                disable=not progress_bar,
                leave=leave):
    
    # Skip tokens that are not in the token_indices list
    if token_indices is not None:
        # Account for the BOS (Beginning of Sentence) token
        if i-1 not in token_indices:
            # Create placeholder tensor on the same device as batch_tokens
            token_probs = torch.zeros(1, len(alphabet), device=device)
            all_token_probs.append(token_probs)
            continue 

    # Mask the tokens
    masked_tokens = batch_tokens.clone()
    masked_tokens[0, i] = mask_token_id
    masked_protein_tensor = attr.evolve(protein_tensor, 
                                        sequence=masked_tokens.squeeze()) 
    # Get logits at the masked position
    with torch.no_grad():
        logits_output = model.logits(
            masked_protein_tensor,
            LogitsConfig(sequence=True)
            )
    # The output logits are 1 x seq_len x 64: this is not the true alphabet size (33)
    # https://github.com/evolutionaryscale/esm/issues/86
    # https://github.com/evolutionaryscale/esm/issues/252
    token_logits = logits_output.logits.sequence[0, i]
    # Convert logits to probabilities
    token_probs = torch.log_softmax(token_logits, dim=-1)

    all_token_probs.append(token_probs.unsqueeze(0))  

# Concatenate all token probabilities along dimension 0 (sequence length)
# and then add a batch dimension (unsqueeze at dim 0)
# This creates a tensor of shape [1, sequence_length, vocab_size]
token_probs = torch.cat(all_token_probs, dim=0).unsqueeze(0)

In [57]:
_check_token_probs(token_probs,
                   batch_tokens,
                   alphabet)

## ESMC

### Local

In [8]:
from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig

protein = ESMProtein(sequence="AAAAA")

client = ESMC.from_pretrained("esmc_300m").to("cuda") # or "cpu"
protein_tensor = client.encode(protein)
logits_output = client.logits(
   protein_tensor, LogitsConfig(sequence=True, return_embeddings=True)
)
print(logits_output.logits, logits_output.embeddings)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

ForwardTrackData(sequence=tensor([[[-38.2500, -38.0000, -38.0000,  12.5625,  21.6250,  22.2500,  22.0000,
           21.8750,  21.5000,  21.6250,  21.7500,  21.3750,  20.7500,  21.5000,
           22.0000,  21.0000,  20.7500,  20.3750,  20.3750,  20.2500,  21.8750,
           19.8750,  19.8750,  19.3750,  18.3750,   1.1875,  -1.7812,  -4.1250,
          -20.7500, -38.0000, -38.0000, -38.2500, -38.0000, -38.2500, -38.2500,
          -38.2500, -38.0000, -38.0000, -38.0000, -38.0000, -38.0000, -38.0000,
          -38.0000, -38.0000, -38.0000, -38.2500, -38.2500, -38.0000, -38.2500,
          -38.0000, -38.2500, -38.0000, -38.0000, -38.2500, -38.0000, -38.0000,
          -38.0000, -38.0000, -38.0000, -38.0000, -38.2500, -38.0000, -38.0000,
          -38.0000],
         [-40.0000, -40.0000, -40.0000,   5.4062,  20.1250,  19.8750,  18.7500,
           20.6250,  18.8750,  18.3750,  18.6250,  18.5000,  18.1250,  18.0000,
           18.7500,  17.7500,  17.3750,  17.1250,  17.7500,  16.8750,  22

### ESM Forge

In [29]:
from getpass import getpass

token = getpass("Token from Forge console: ")

In [ ]:
# Model	Model Size	Number of Layers	Release Date
# esmc-6b-2024-12	6B	80	2024-12
# esmc-600m-2024-12	600M	36	2024-12
# esmc-300m-2024-12	300M	30	2024-12

model_name = "esmc_300m" #"esmc-300m-2024-12" #

## Remote
run_local = True
if not run_local:
    from esm.sdk import client
    model = client(
        model=model_name, 
        url="https://forge.evolutionaryscale.ai", 
        token=token
    )
## Local
else:
    import torch
    from esm.models.esmc import ESMC
    torch.cuda.set_device(1)
    model = ESMC.from_pretrained(model_name).to("cuda")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [5]:
from concurrent.futures import ThreadPoolExecutor
from typing import Sequence

from esm.sdk.api import (
    ESM3InferenceClient,
    ESMProtein,
    ESMProteinError,
    LogitsConfig,
    LogitsOutput,
    ProteinType,
)

EMBEDDING_CONFIG = LogitsConfig(
    sequence=True, 
    return_embeddings=True, 
    return_hidden_states=True
)


def embed_sequence(model: ESM3InferenceClient, sequence: str) -> LogitsOutput:
    protein = ESMProtein(sequence=sequence)
    protein_tensor = model.encode(protein)
    output = model.logits(protein_tensor, EMBEDDING_CONFIG)
    return output


def embed_sequence_parallel(
    model: ESM3InferenceClient, 
    inputs: Sequence[ProteinType],
    max_workers: int = 1,
    error: bool = False
) -> Sequence[LogitsOutput]:
    """Forge supports auto-batching. So batch_embed() is as simple as running a collection
    of embed calls in parallel using asyncio.
    """
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(embed_sequence, model, protein) for protein in inputs
        ]
        results = []
        for future in futures:
            try:
                results.append(future.result())
            except Exception as e:
                if error:
                    raise e
                else:
                    results.append(ESMProteinError(500, str(e)))
    return results

## Read in haplotypes

In [ ]:
tx_ids = hs.list_haplotypes()
hap_df = ESM3.haplosaurus_dataloader(tx_ids=tx_ids[:10])

In [ ]:
outputs = embed_sequence_parallel(model=model, 
                                  inputs=hap_df["sequence"].tolist())
outputs

## ESM++

https://huggingface.co/Synthyra/ESMplusplus_small

https://github.com/evolutionaryscale/esm/issues/176#issuecomment-2568427933


In [7]:
# Define paths
model_name = "ESM2-3B"
base_dir = "/home/schilder/projects/data/1000_Genomes_on_GRCh38"

haplotype_embeddings_dir = os.path.join(base_dir,
                                         "haplotype_embeddings/", 
                                         model_name)
print(haplotype_embeddings_dir)


# Load model
import torch
from transformers import AutoModelForMaskedLM #AutoModel also works

torch.cuda.set_device(1)
model = AutoModelForMaskedLM.from_pretrained('Synthyra/'+model_name,
                                              trust_remote_code=True)
model.to("cuda")
print()

# Get list of tx_ids
tx_ids = hs.list_haplotypes()
# tx_ids.reverse()

In [ ]:
ESM3.embed_sequences(model = model,
                      save_dir = haplotype_embeddings_dir, 
                      tx_ids = tx_ids,
                      batch_size = 25)

### Get total size of embeddings

In [5]:
# Get total number of files in the directory
!ls -l {haplotype_embeddings_dir} | wc -l

47326


In [ ]:
# Get total size of embeddings
!du -sh {haplotype_embeddings_dir}

15G	/home/schilder/projects/data/1000_Genomes_on_GRCh38/embeddings/ESMplusplus_large/


In [11]:
# Get all sample names
samples = hs.get_haplotype_samples(max_tx_ids=10,
                                   unnest=True,
                                    cohort='1000GENOMES:phase_3',
                                    remove_prefix=True,
                                    key='protein_haplotypes')

# Split samples into batches
sample_batches = utils.split_batches(samples, 
                                     batch_size=3)

Found haplotypes of 47325 transcripts in: '/home/schilder/.cache/ensembl_rest/haplotypes'


Getting haplotypes:   0%|          | 0/10 [00:00<?, ?it/s]

Extracting sample IDs:   0%|          | 0/10 [00:00<?, ?it/s]

Split 2504 samples into 835 batches of ~3


In [ ]:
base_dir = "/home/schilder/projects/data/1000_Genomes_on_GRCh38"
model_name = "ESMplusplus_small"
haplotype_embeddings = os.path.join(base_dir,
                                    "haplotype_embeddings", 
                                    model_name)
patient_embeddings_dir = os.path.join(base_dir,
                                      "patient_embeddings", 
                                      model_name)

from tqdm import tqdm
for batch in tqdm(sample_batches):
    save = ESM3.make_patient_tensor(haplotype_embeddings=haplotype_embeddings,
                                    patient_embeddings_dir=patient_embeddings_dir,
                                    samples=batch,
                                    verbose=False,
                                    tx_ids=hs.list_haplotypes()[:10]
                                )
    

In [22]:
import glob
tensor_list = []
for path in glob.glob(os.path.join(patient_embeddings_dir, "*.pth"))[:10]:
    print(path)
    tensor_list.append(torch.load(path))
# tensor = torch.cat(tensor_list, dim=0)



/home/schilder/projects/data/1000_Genomes_on_GRCh38/patient_embeddings/ESMplusplus_small/HG02511.pth
/home/schilder/projects/data/1000_Genomes_on_GRCh38/patient_embeddings/ESMplusplus_small/NA12762.pth
/home/schilder/projects/data/1000_Genomes_on_GRCh38/patient_embeddings/ESMplusplus_small/HG02759.pth
/home/schilder/projects/data/1000_Genomes_on_GRCh38/patient_embeddings/ESMplusplus_small/HG01440.pth
/home/schilder/projects/data/1000_Genomes_on_GRCh38/patient_embeddings/ESMplusplus_small/NA19068.pth
/home/schilder/projects/data/1000_Genomes_on_GRCh38/patient_embeddings/ESMplusplus_small/HG03886.pth
/home/schilder/projects/data/1000_Genomes_on_GRCh38/patient_embeddings/ESMplusplus_small/NA12286.pth
/home/schilder/projects/data/1000_Genomes_on_GRCh38/patient_embeddings/ESMplusplus_small/HG02385.pth
/home/schilder/projects/data/1000_Genomes_on_GRCh38/patient_embeddings/ESMplusplus_small/NA19057.pth
/home/schilder/projects/data/1000_Genomes_on_GRCh38/patient_embeddings/ESMplusplus_small/HG